# Notebook 03 — Model Training

In this notebook, I will train object detection models for road damage detection.

The models for comparison are:

- YOLOv8 — One-stage CNN-based detector
- Faster R-CNN — Two-stage detector
- RT-DETR — Transformer-based detector

## Imports

In [1]:
!nvidia-smi

Sun Aug 30 10:54:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 5.3 MB/s eta 0:00:00


In [3]:
from ultralytics import YOLO
from pathlib import Path
import yaml

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [4]:
!git clone https://github.com/gracyy-rm/Road_damage_object_detection.git

Cloning into 'Road_damage_object_detection'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 25 (delta 2), reused 24 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 9.11 MiB | 30.79 MiB/s, done.
Resolving deltas: 100% (2/2), done.


## Project Paths

In [5]:
PROJECT_ROOT = Path("/kaggle/working/Road_damage_object_detection")
CONFIG_DIR = PROJECT_ROOT / "configs"
DATA_YAML = CONFIG_DIR / "rdd2022.yaml"
YOLO_CONFIG = CONFIG_DIR / "yolov8_config.yaml"

print(YOLO_CONFIG.exists())

True


In [6]:
%cd '/kaggle/working/Road_damage_object_detection'

/kaggle/working/Road_damage_object_detection


In [7]:
!pwd

/kaggle/working/Road_damage_object_detection


## Loading Training Configuration


In [8]:
with open(YOLO_CONFIG ) as f:
    yolo_config = yaml.safe_load(f)
yolo_config

{'model': 'yolov8s.pt',
 'imgsz': 640,
 'batch': 32,
 'epochs': 50,
 'patience': 10,
 'device': 0,
 'seed': 42,
 'pretrained': True,
 'plots': True}

## YOLOv8 Training Settings

In [9]:
MODEL_NAME = yolo_config["model"]
IMG_SIZE = yolo_config["imgsz"]
BATCH_SIZE = yolo_config["batch"]
EPOCHS = yolo_config["epochs"]
PATIENCE = yolo_config["patience"]
DEVICE = yolo_config["device"]
SEED = yolo_config["seed"]

print("Model:", MODEL_NAME)
print("Image Size:", IMG_SIZE)
print("Batch Size:", BATCH_SIZE)
print("Max Epochs:", EPOCHS)
print("Patience:", PATIENCE)
print("Device:", DEVICE)

Model: yolov8s.pt
Image Size: 640
Batch Size: 32
Max Epochs: 50
Patience: 10
Device: 0


In [10]:
print(DATA_YAML.read_text())

path: /kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT

train: train/images
val: val/images
test: test/images

nc: 5

names:
  0: longitudinal crack
  1: transverse crack
  2: alligator crack
  3: other corruption
  4: pothole


## Loading the Pretrained YOLOv8 Model

In [11]:
yolov8_model = YOLO(MODEL_NAME)

## Model Overview

In [12]:
yolov8_model.info()

YOLOv8s summary: 129 layers, 11,166,560 parameters, 0 gradients, 28.8 GFLOPs


(129, 11166560, 0, 28.8119296)

In [13]:
yolov8_model.task

'detect'

## BaseLine Training

In [14]:
results = yolov8_model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(PROJECT_ROOT / "outputs"),
    name="yolov8s_50epochs",
    pretrained=yolo_config["pretrained"],
    plots=yolo_config["plots"],
    seed=SEED
)

Ultralytics 8.4.135 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Road_damage_object_detection/configs/rdd2022.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, n

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50      6.73G       1.93        2.3       1.66         50        640: 100% ━━━━━━━━━━━━ 840/840 1.8it/s 7:38
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 90/90 1.5it/s 58.5s
                   all       5758       9740      0.405      0.273      0.227     0.0977

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      6.73G      2.013      2.456      1.747         63        640: 100% ━━━━━━━━━━━━ 840/840 1.9it/s 7:22
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 90/90 1.5it/s 58.1s
                   all       5758       9740       0.26      0.215      0.159     0.0619

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      6.77G      2.042      2.503      1.797         78        640: 100% ━━━━━━━━━━━━ 840/840 1.9it/s 7:21
                 Class     Images  Instances      Box(